# GraphSAGE 32-dim Embeddings — Dataset 1 p-values & Dataset 2

Uses the same GraphSAGE v1 32-dim config as the best model from g1.  
Embeddings are graph-structural (do not depend on p), so they are computed **once** per dataset and merged with each p-value target.

In [1]:
import sys
import os
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.models.embeddings import GNNConfig, extract_embeddings

PROJECT_ROOT = find_project_root()
EMB_DIR      = PROJECT_ROOT / 'src' / 'data' / 'embeddings'
TARGETS_DIR  = PROJECT_ROOT / 'src' / 'datasets' / 'targets'
DATASET2_PATH = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_2'

EMB_DIR.mkdir(parents=True, exist_ok=True)

P_VALUES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

## Config

In [2]:
# Same config as g1_ref graphsage_v1_32
cfg = GNNConfig(hidden_dims=(256, 32), dropout=0.3, lr=0.01, epochs=100, aggregation='mean', device='cpu')

## Dataset 1 — p-value targets

Embeddings are already computed in `graphsage_v1_32_srisk_dataset.parquet`.  
For each p value, drop the baseline target and merge the p-specific one.

In [3]:
baseline = pd.read_parquet(EMB_DIR / 'graphsage_v1_32_srisk_dataset.parquet')
emb_cols = [c for c in baseline.columns if c.startswith('emb_')]
base_keys = ['bank_id', 'year', 'quarter', 'period']
embeddings_d1 = baseline[base_keys + emb_cols].copy()

print(f'Baseline embeddings loaded: {embeddings_d1.shape}')

Baseline embeddings loaded: (145536, 36)


In [4]:
for p in P_VALUES:
    p_label = f'p{int(p * 100)}'
    p_dir = TARGETS_DIR / p_label

    # Load all quarterly targets for this p value
    frames = []
    for csv_path in sorted(p_dir.glob('target_*.csv')):
        df = pd.read_csv(csv_path)
        period = csv_path.stem.replace('target_', '')
        df['period'] = period
        frames.append(df)

    if not frames:
        print(f'{p_label}: no target files found, skipping')
        continue

    targets_p = pd.concat(frames, ignore_index=True)
    merged = embeddings_d1.merge(targets_p, on=['bank_id', 'period'], how='inner')

    out_path = EMB_DIR / f'graphsage_v1_32_{p_label}_dataset.parquet'
    merged.to_parquet(out_path, index=False)
    print(f'{p_label}: shape={merged.shape}  -> {out_path.name}')

p5: shape=(145536, 38)  -> graphsage_v1_32_p5_dataset.parquet
p10: shape=(145536, 38)  -> graphsage_v1_32_p10_dataset.parquet
p15: shape=(145536, 38)  -> graphsage_v1_32_p15_dataset.parquet
p20: shape=(145536, 38)  -> graphsage_v1_32_p20_dataset.parquet
p25: shape=(145536, 38)  -> graphsage_v1_32_p25_dataset.parquet
p30: shape=(145536, 38)  -> graphsage_v1_32_p30_dataset.parquet
p35: shape=(145536, 38)  -> graphsage_v1_32_p35_dataset.parquet
p40: shape=(145536, 38)  -> graphsage_v1_32_p40_dataset.parquet


## Dataset 2 — compute embeddings once, merge all targets

In [5]:
def load_dataset2_nodes(dataset_path):
    nodes = pd.read_csv(dataset_path / 'nodes.csv')
    nodes = nodes.reset_index(drop=True)
    nodes['index']  = nodes.index
    nodes['Equity'] = nodes['buffer']
    nodes['Assets'] = nodes['assets']
    return nodes


def load_dataset2_edges(dataset_path, bank_to_idx):
    matrix = pd.read_excel(dataset_path / 'network.xlsx', index_col=0)
    edges_long = matrix.stack().reset_index()
    edges_long.columns = ['source_bank', 'target_bank', 'Weights']
    edges_long = edges_long[edges_long['Weights'] != 0].copy()
    edges_long['Sourceid'] = edges_long['source_bank'].map(bank_to_idx)
    edges_long['Targetid'] = edges_long['target_bank'].map(bank_to_idx)
    edges_long = edges_long.dropna(subset=['Sourceid', 'Targetid'])
    edges_long['Sourceid'] = edges_long['Sourceid'].astype(int)
    edges_long['Targetid'] = edges_long['Targetid'].astype(int)
    return edges_long[['Sourceid', 'Targetid', 'Weights']].reset_index(drop=True)

In [6]:
nodes_d2 = load_dataset2_nodes(DATASET2_PATH)
bank_to_idx_d2 = dict(zip(nodes_d2['bank'], nodes_d2['index']))
edges_d2 = load_dataset2_edges(DATASET2_PATH, bank_to_idx_d2)

# Use only core node features. Drop stress/loss/default outcome columns.
feature_cols_d2 = ['assets', 'liabilities', 'buffer']

print(f'Dataset 2: {len(nodes_d2)} banks, {len(edges_d2)} edges')
print('Training GraphSAGE...')
emb_df_d2, _ = extract_embeddings(edges_d2, nodes_d2, cfg, feature_cols=feature_cols_d2)
embedding_cols_d2 = [c for c in emb_df_d2.columns if c.startswith('emb_')]
final_base_cols_d2 = ['bank_id'] + embedding_cols_d2
print(f'Embeddings shape: {emb_df_d2.shape}')

Dataset 2: 1444 banks, 2893 edges
Training GraphSAGE...
Embeddings shape: (1444, 33)


In [7]:
d2_targets_dir = DATASET2_PATH / 'targets'

# Baseline (p=100%)
target_baseline = pd.read_csv(d2_targets_dir / 'target.csv')
target_cols = [c for c in target_baseline.columns if c != 'bank_id']
merged_baseline = emb_df_d2[final_base_cols_d2].merge(target_baseline, on='bank_id', how='inner')
merged_baseline = merged_baseline[final_base_cols_d2 + target_cols]
out_path = EMB_DIR / 'graphsage_v1_32_dataset2_dataset.parquet'
merged_baseline.to_parquet(out_path, index=False)
print(f'baseline : shape={merged_baseline.shape}  -> {out_path.name}')

# p-values
for p in P_VALUES:
    p_label = f'p{int(p * 100)}'
    target_path = d2_targets_dir / f'target_{p_label}.csv'
    if not target_path.exists():
        print(f'{p_label}: target file not found, skipping')
        continue
    target_p = pd.read_csv(target_path)
    target_cols = [c for c in target_p.columns if c != 'bank_id']
    merged = emb_df_d2[final_base_cols_d2].merge(target_p, on='bank_id', how='inner')
    merged = merged[final_base_cols_d2 + target_cols]
    out_path = EMB_DIR / f'graphsage_v1_32_dataset2_{p_label}_dataset.parquet'
    merged.to_parquet(out_path, index=False)
    print(f'{p_label}     : shape={merged.shape}  -> {out_path.name}')

baseline : shape=(1444, 35)  -> graphsage_v1_32_dataset2_dataset.parquet
p5     : shape=(1444, 35)  -> graphsage_v1_32_dataset2_p5_dataset.parquet
p10     : shape=(1444, 35)  -> graphsage_v1_32_dataset2_p10_dataset.parquet
p15     : shape=(1444, 35)  -> graphsage_v1_32_dataset2_p15_dataset.parquet
p20     : shape=(1444, 35)  -> graphsage_v1_32_dataset2_p20_dataset.parquet
p25     : shape=(1444, 35)  -> graphsage_v1_32_dataset2_p25_dataset.parquet
p30     : shape=(1444, 35)  -> graphsage_v1_32_dataset2_p30_dataset.parquet
p35     : shape=(1444, 35)  -> graphsage_v1_32_dataset2_p35_dataset.parquet
p40     : shape=(1444, 35)  -> graphsage_v1_32_dataset2_p40_dataset.parquet


## Output Summary

In [8]:
files = sorted(EMB_DIR.glob('graphsage_v1_32_p*.parquet')) + \
        sorted(EMB_DIR.glob('graphsage_v1_32_dataset2*.parquet'))

for f in files:
    df = pd.read_parquet(f)
    print(f'{f.name:55s}  shape={df.shape}')

graphsage_v1_32_p10_dataset.parquet                      shape=(145536, 38)
graphsage_v1_32_p15_dataset.parquet                      shape=(145536, 38)
graphsage_v1_32_p20_dataset.parquet                      shape=(145536, 38)
graphsage_v1_32_p25_dataset.parquet                      shape=(145536, 38)
graphsage_v1_32_p30_dataset.parquet                      shape=(145536, 38)
graphsage_v1_32_p35_dataset.parquet                      shape=(145536, 38)
graphsage_v1_32_p40_dataset.parquet                      shape=(145536, 38)
graphsage_v1_32_p5_dataset.parquet                       shape=(145536, 38)
graphsage_v1_32_dataset2_dataset.parquet                 shape=(1444, 35)
graphsage_v1_32_dataset2_p10_dataset.parquet             shape=(1444, 35)
graphsage_v1_32_dataset2_p15_dataset.parquet             shape=(1444, 35)
graphsage_v1_32_dataset2_p20_dataset.parquet             shape=(1444, 35)
graphsage_v1_32_dataset2_p25_dataset.parquet             shape=(1444, 35)
graphsage_v1_32_datase